In [ ]:
#This part does not work all the way and I couldnt figure it out

from google.colab import files
import nltk
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, LSTM, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

uploaded = files.upload()

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

data = pd.read_csv('IMDB-Movie-Data.csv')
print(data.columns)
print(data['Title'][1])
print(data['Genre'][1])
print(data['Description'][1])

stop_words = set(stopwords.words('english'))
print(stop_words)
lemmatizer = WordNetLemmatizer()

def preprocess(text):
  if not isinstance(text,str) or pd.isna(text) or text =='':
    return ''
  tokens = nltk.word_tokenize(text.lower())
  filtered_tokens = []
  for token in tokens:
    if token not in stop_words and token.isalnum():
      filtered_tokens.append(lemmatizer.lemmatize(token))
  return ' '.join(filtered_tokens)

data['clean_desc'] = data['Description'].fillna('').apply(preprocess)

genres = data['Genre'].apply(lambda x: x.split(', '))
print(genres)
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(genres)
print(genres_encoded[0])
print(mlb.classes_)

maxW = 10000
maxL = 100

tokenizer = Tokenizer(num_words = maxW)
tokenizer.fit_on_texts(data['clean_desc'])
X = pad_sequences(tokenizer.texts_to_sequences(data['clean_desc']), maxlen = maxL)

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

embeddings_index = {}
with open('glove.6B.100d.txt', encoding = 'utf-8') as f:
  for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs

print(f'Found {len(embeddings_index)} word vectors.')

embedding_dim = 100
embedding_matrix = np.zeros((maxW, embedding_dim))
for word, i in tokenizer.word_index.items():
  if i < maxW:
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
      embedding_matrix[i] = embedding_vector

xTrain, xVal, xTest = X[:700], X[700:800], X[800:1000]
yTrain, yVal, yTest = genres_encoded[:700], genres_encoded[700:800], genres_encoded[800:1000]

model_ff = Sequential([
    Embedding(maxW, embedding_dim, weights = [embedding_matrix], trainable = False),
    Flatten(),
    Dense(128, activation = 'relu'),
    Dropout(.5),
    Dense(20, activation = 'sigmoid')
])

model_ff.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
history_ff = model_ff.fit(xTrain, yTrain, validation_data=(xVal, yVal), epochs=20, batch_size=32)


model_lstm = Sequential([
     Embedding(maxW, embedding_dim, weights = [embedding_matrix], trainable = False),
     LSTM(128, return_sequences=False),
     Dropout(.5),
     Dense(20, activation ='sigmoid')
])

model_lstm.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
history_lstm = model_lstm.fit(xTrain, yTrain, validation_data=(xVal, yVal), epochs=20, batch_size=32)

def evaluate_model(model, xTest, yTest):
  yPred = (model.predict(xTest) > .5).astype(int)
  acc = accuracy_score(yTest, yPred)
  prec = precision_score(yTest, yPred, average='macro', zero_division=0)
  rec = recall_score(yTest, yPred, average='macro', zero_division=0)
  return acc, prec, rec


print("Feed Forward - Accuracy: ", evaluate_model(model_ff, xTest, yTest))
print("LSTM - Accuracy: ", evaluate_model(model_lstm, xTest, yTest))

data['combined'] = data['Title'] + ' ' + data['clean_desc']
tokenizer_titles = Tokenizer(num_words = maxW)
tokenizer_titles.fit_on_texts(data['combined'])
x_combined = pad_sequences(tokenizer_titles.texts_to_sequences(data['combined']), maxlen = maxL)

xTrain_c, xVal_c, xTest_c = x_combined[:700], x_combined[700:800], x_combined[800:1000]

model_titles = Sequential([
    Embedding(maxW, embedding_dim, weights=[embedding_matrix], trainable=False),
    LSTM(128),
    Dropout(.5),
    Dense(20, activation = 'sigmoid')
])

model_titles.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
model_titles.fit(xTrain_c, yTrain, validation_data=(xVal_c, yVal), epochs = 20, batch_size = 32)

print("With Titles - Accuracy: ", evaluate_model(model_titles, xTest_c, yTest))





Saving IMDB-Movie-Data.csv to IMDB-Movie-Data (2).csv
Index(['Rank', 'Title', 'Genre', 'Description', 'Director', 'Actors', 'Year',
       'Runtime (Minutes)', 'Rating', 'Votes', 'Revenue (Millions)',
       'Metascore'],
      dtype='object')
Prometheus
Adventure,Mystery,Sci-Fi
Following clues to the origin of mankind, a team finds a structure on a distant moon, but they soon realize they are not alone.
{"he'd", 'aren', 'hasn', 'on', "we'll", 'above', "it'll", 'd', "he'll", "you'll", "you'd", 'themselves', 'then', 'were', 'same', 'needn', 'are', "couldn't", 'how', 'nor', 'yourselves', 'off', 're', 'had', "mightn't", 'these', 'ours', 'them', 'has', 'this', 'doesn', "doesn't", 'to', 'because', 'does', 'whom', 'do', 'of', "they're", "don't", 'been', "hasn't", "i'd", 'just', 'from', "i'm", "isn't", 'no', 'haven', "she's", 'theirs', 'than', 'very', 'they', 'too', 'it', "won't", 'won', 'if', 'mustn', 've', 'we', 'other', 'their', 'our', "aren't", 'most', 'in', 'he', 'so', 'below', 'hadn', '

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


0       [Action,Adventure,Sci-Fi]
1      [Adventure,Mystery,Sci-Fi]
2               [Horror,Thriller]
3       [Animation,Comedy,Family]
4      [Action,Adventure,Fantasy]
                  ...            
995         [Crime,Drama,Mystery]
996                      [Horror]
997         [Drama,Music,Romance]
998            [Adventure,Comedy]
999       [Comedy,Family,Fantasy]
Name: Genre, Length: 1000, dtype: object
[0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
['Action' 'Action,Adventure' 'Action,Adventure,Biography'
 'Action,Adventure,Comedy' 'Action,Adventure,Crime'
 'Action,Adventure,Drama' 'Action,Adventure,Family'
 '

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 207), output.shape=(None, 20)

In [ ]:
#This one works but I only tested with 1 epochs not with any more than that due to time for it to load

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

import kagglehub
path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Path to dataset files:", path)

train_dir = os.path.join(path, 'seg_train/seg_train')
val_dir = os.path.join(path, 'seg_test/seg_test')

train_datagen = ImageDataGenerator(rescale = 1./255)
val_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(150,150), batch_size=32, class_mode='categorical'
)


val_generator = val_datagen.flow_from_directory(
    train_dir, target_size=(150,150), batch_size=32, class_mode='categorical'
)

model_cnn_3 = Sequential([
    Conv2D(32, (3,3), activation ='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3), activation ='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation = 'relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dropout(.5),
    Dense(512, activation= 'relu'),
    Dense(6, activation = 'relu')
])

model_cnn_3.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics=['accuracy'])
history_3 = model_cnn_3.fit(train_generator, validation_data=val_generator, epochs=3)

model_cnn_6 = Sequential([
     Conv2D(32, (3,3), activation ='relu', padding= 'same', input_shape=(150,150,3)),
     Conv2D(32, (3,3), activation ='relu', padding= 'same'),
     MaxPooling2D(2,2),
     Conv2D(64, (3,3), activation ='relu', padding= 'same'),
     Conv2D(64, (3,3), activation ='relu', padding= 'same'),
     MaxPooling2D(2,2),
     Conv2D(128, (3,3), activation ='relu', padding= 'same'),
     Conv2D(128, (3,3), activation ='relu', padding= 'same'),
     MaxPooling2D(2,2),
     Flatten(),
     Dropout(.5),
     Dense(512, activation = 'relu'),
     Dense(6, activation = 'softmax')

])

model_cnn_6.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics=['accuracy'])
history_6 = model_cnn_6.fit(train_generator, validation_data = val_generator, epochs= 20)

print("Model 1 with 3 layers - Validation Accuracy: ", max(history_3.history['val_accuracy']))
print("Model 2 with 6 layers - Validation Accuracy: ", max(history_6.history['val_accuracy']))



Using Colab cache for faster access to the 'intel-image-classification' dataset.
Path to dataset files: /kaggle/input/intel-image-classification
Found 14034 images belonging to 6 classes.
Found 14034 images belonging to 6 classes.
Epoch 1/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 44s 92ms/step - accuracy: 0.3305 - loss: 2.0561 - val_accuracy: 0.1967 - val_loss: 1.7825
Epoch 2/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 88ms/step - accuracy: 0.3633 - loss: 1.5923 - val_accuracy: 0.5222 - val_loss: 1.4030
Epoch 3/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 86ms/step - accuracy: 0.4974 - loss: 1.4272 - val_accuracy: 0.3487 - val_loss: 1.5124
439/439 ━━━━━━━━━━━━━━━━━━━━ 58s 123ms/step - accuracy: 0.5980 - loss: 0.9848 - val_accuracy: 0.7272 - val_loss: 0.7063
Model 1 with 3 layers - Validation Accuracy:  0.5221604704856873
Model 2 with 6 layers - Validation Accuracy:  0.7271625995635986
